In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

dap_url = (
    "https://geoport.whoi.edu/thredds/dodsC/"
    "vortexfs1/usgs/Projects/Helene2024/helene77/Output_89pct/"
    "ww3.202409_spec.nc"
)

ds = xr.open_dataset(dap_url, engine="netcdf4")  # netCDF4 is the usual OPeNDAP backend
ds

<xarray.Dataset> Size: 238MB
Dimensions:       (time: 121, station: 425, string40: 40, frequency: 32,
                   direction: 36)
Coordinates:
  * time          (time) datetime64[ns] 968B 2024-09-24 ... 2024-09-29
  * station       (station) float64 3kB 1.0 2.0 3.0 4.0 ... 423.0 424.0 425.0
  * string40      (string40) float64 320B nan nan nan nan ... nan nan nan nan
  * frequency     (frequency) float32 128B 0.038 0.0418 ... 0.6631 0.7294
  * direction     (direction) float32 144B 90.0 80.0 70.0 ... 120.0 110.0 100.0
Data variables:
    station_name  (station) |S64 27kB ...
    frequency1    (frequency) float32 128B ...
    frequency2    (frequency) float32 128B ...
    longitude     (time, station) float32 206kB ...
    latitude      (time, station) float32 206kB ...
    efth          (time, station, frequency, direction) float32 237MB ...
    dpt           (time, station) float32 206kB ...
    wnd           (time, station) float32 206kB ...
    wnddir        (time, station) float32 206kB ...
    cur           (time, station) float32 206kB ...
    curdir        (time, station) float32 206kB ...
Attributes: (12/19)
    product_name:                    ww3.202409_spec.nc
    area:                            H Ian
    data_type:                       OCO spectra 2D
    format_version:                  1.1
    southernmost_latitude:           n/a
    northernmost_latitude:           n/a
    ...                              ...
    start_date:                      2024-09-24 00:00:00
    stop_date:                       2024-09-29 00:00:00
    field_type:                      n/a
    DODS.strlen:                     40
    DODS.dimName:                    string40
    DODS_EXTRA.Unlimited_Dimension:  time

In [2]:
freq = ds['frequency'].values
theta = np.deg2rad( ds['direction'].values )
dtheta = np.abs( theta[1]-theta[0] )
print(dtheta)

efth = ds['efth'].values   # (time, station, frequency, direction)

# integrate over direction (rad) -> m^2 s
E_f = np.sum(efth, axis=-1)*dtheta

# integrate over frequency (Hz) -> m^2
m0 = np.trapezoid(E_f, freq, axis=-1)

# calculate Hm0
Hm0 = 4 * np.sqrt(m0)

print("m0 min/max:", np.min(m0), np.max(m0))
print("Hm0 min/max:", np.min(Hm0), np.max(Hm0))

0.17453301
m0 min/max: 0.00024312836 7.5354195
Hm0 min/max: 0.062370297 10.980288
